# Structured output
### Pydantic


In [9]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026191588710>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002619158A7B0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="Title of the Movie")
    year:int = Field(description="This year the Movie was released")
    director:str = Field(description="The Director of the movie")
    rating:float = Field(description="The ratings of the movie out of 10")
    uniqness:str = Field(description="The Uniqueness of the movie")


In [11]:
model_with_structure = model.with_structured_output(Movie)

In [12]:
# res = model.invoke("GodFather")
# print(res)

In [13]:
# structured_res = model_with_structure.invoke("Matrix")
# print(structured_res)

In [14]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(description="Name of the actor")
    role: str = Field(description="Name of the character they played")

class MovieDetails(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Release year")
    cast: list[Actor] = Field(description="Main cast members and their roles")
    genres: list[str] = Field(description="Genres of the movie")
    budget: float | None = Field(None, description="Budget in millions USD")

# 3. Execution code goes OUTSIDE the class definitions
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Tell me about the Movie of The Godfather")

print(response)

title='The Godfather' year=1972 cast=[Actor(name='Marlon Brando', role='Vito Corleone'), Actor(name='Al Pacino', role='Michael Corleone'), Actor(name='James Caan', role='Sonny Corleone'), Actor(name='Robert Duvall', role='Tom Hagen'), Actor(name='Diane Keaton', role='Kay Adams'), Actor(name='Talia Shire', role='Connie Corleone'), Actor(name='John Marley', role='Jack Woltz'), Actor(name='Richard S. Castellano', role='Clemenza')] genres=['Crime', 'Drama'] budget=None


In [15]:
from typing_extensions import TypedDict, Annotated

class movieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, "The title of the Movie"]
    year: Annotated[int, "The year of the Movie released"]
    director: Annotated[str, "The director of the Movie"]
    rating: Annotated[float, "The rating of the Movie"]
    Uniqueness: Annotated[str, "The uniqueness of the Movie"]

# 3. Bind to the model and invoke
typed_with_structure = model.with_structured_output(movieDict)

res = typed_with_structure.invoke("give context overview of The Notebook")
print(res)

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '**The Notebook – Context Overview**\n\n**1. Basic Information**  \n- **Title:** *The Notebook*  \n- **Release Year:** 2004  \n- **Director:** Nick Cassavetes  \n- **Screenplay:** Based on the 1996 novel *The Notebook* by Nicholas Sparks (adapted by Jeremy Leven and Jan Sardi)  \n- **Principal Cast:** Ryan Gosling (Noah Calhoun), Rachel McAdams (Allie Hamilton), James Garner (Older Noah), Gena Rowlands (Older Allie)  \n- **Genre:** Romantic drama, period romance  \n\n**2. Plot Synopsis (High‑Level)**\nThe film tells a sweeping love story that unfolds across two time periods:\n\n| Timeline | Core Events |\n|----------|-------------|\n| **1940s (Young Love)** | In a small Southern town, Noah (a working‑class laborer) and Allie (a wealthy, spirited young woman) meet one summer and fall passionate